Predicting Customer Propensity to Buy
In this example we'll use clicks on a website (the data from clicks) when someone is browsing to predict their propensity to buy the product.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import seaborn as sn

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
import sklearn.metrics

#pd.set_option('display.float_format', lambda x: '%.5f' % x)

In [2]:
prospect_data = pd.read_csv('online_behaviour.csv')
prospect_data.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'online_behaviour.csv'

In [ ]:
prospect_data.dtypes

In [ ]:
prospect_data.describe()

In [ ]:
#Drop any irrelavant columns. 
prospect_data.drop(['UserID'], axis =1, inplace = True)
prospect_data.head()
#corr().head()

In [ ]:
#To determine which columns are related to 'BUY'. (Ordered target column & highest correlation)
correlation = prospect_data.corr()['purchased']
correlation.nlargest(10).round(2)

In [ ]:
#to display correlations using a heatmap using Seaborn library.
f,ax = plt.subplots(figsize=(10,8))

#Creating a variable
corr = prospect_data.corr()
sn.heatmap(corr.round(2), annot=True, vmin =-1, vmax=1, center=0,cmap="vlag")

In [ ]:
#to display correlations using a heatmap
matrix = prospect_data.corr().round(2)
#use Mask funtion to improve masking.
mask = np.triu(np.ones_like(matrix,dtype=bool))
mask

In [ ]:
sn.heatmap(matrix, annot=True, vmin =-1, vmax=1, center=0,cmap="vlag", mask=mask)

In [ ]:
matrix['purchased'].nlargest(7)

In [ ]:
matrix[matrix >0]

In [ ]:
#Change the colour. Use lamba to find all X.  Lamba = An anonymous function which we can pass in instantly wohtout creating a function.
corr.style.map(lambda x: 'color: red'
                              if (x>0.1) and x!=1 
                              else 'color: white')

In [ ]:
# use Seaborn regplot to show a scatterplot of highest correlated value. 
sn.regplot(x=prospect_data["new_releases"], y=prospect_data["purchased"], line_kws={"color": "yellow"})
sn.regplot(x=prospect_data["subscribe_save"], y=prospect_data["purchased"], line_kws={"color": "blue"})
#sn.regplot(x=prospect_data["top_banner_image_click"], y=prospect_data["purchased"], line_kws={"color": "red"})

In [ ]:
#Looking at correlations - 'Reviews','Compare_similar', 'Warranty','Sponsored Links' have medium correlation to BUY.
predictors = prospect_data[['new_releases','subscribe_save','add_to_basket','repeat_purchase','frequently_bought_together']]
targets = prospect_data.purchased
predictors.head(3)

# Training & Testing Split

In [ ]:
#Construct an algorithm that can learn and make preictions in a task. 
#To train and test the data - use ratio of 70:30 (70% of the data is used to train model; 30% to test model).
pred_train, pred_test, tar_train, tar_test = train_test_split(predictors, targets, test_size=.3)
print('Predictor - Training: ',pred_train.shape, 'Predictor - Testing: ', pred_test.shape)

In [ ]:
#Use a preexisting naive bayes classifier to create a matrix array.

from sklearn.naive_bayes import GaussianNB
classifier = GaussianNB()
classifier = classifier.fit(pred_train.values, tar_train.values)
predictions = classifier.predict(pred_test.values)

sklearn.metrics.confusion_matrix(tar_test, predictions)

In [ ]:
#To test the accuracy of the prediction (=73%)
sklearn.metrics.accuracy_score (tar_test, predictions)


In [ ]:
#The probability that the prospect will BUY the product(=15% chance of BUYING).
#So this is the probability when clicks on any 1 column.
pred_prob = classifier.predict_proba(pred_test.values)
pred_prob[0,1].round(2)

In [ ]:
#Real time predictions - customer does not visit ANY of the 4 pages 
#(only 4% prediction they will BUY). The array = [reviews,compare similiar,warranty, sponsored links]

no_clicks = np.array([0,0,0,0]).reshape(1,-1)
print('New visitor - propensity to buy: ', classifier.predict_proba(no_clicks)[:,1].round(2))

In [ ]:
#Real time predictions - customer clicks ONLY on 'Reviews' of the 4 pages. 
#(only 25% prediction they will BUY). The array = [reviews,compare similiar,warranty, sponsored links]

reviews_clicked = np.array([1,0,0,0]).reshape(1,-1)
print('New visitor - propensity to buy: ', classifier.predict_proba(reviews_clicked)[:,1].round(2))

In [ ]:
#Clicking both 'Reviews' & 'Compare Similar' (45% of BUYING)
review_similar = np.array([1,1,0,0]).reshape(1,-1)
print('New visitor - propensity to buy: ', classifier.predict_proba(review_similar)[:,1].round(2))

In [ ]:
#Clicking both 'Reviews' & 'Compare Similar' & Warranty'  (66% of BUYING)
review_similar_warranty = np.array([1,1,1,0]).reshape(1,-1)
print('New visitor - propensity to buy: ', classifier.predict_proba(review_similar_warranty)[:,1].round(2))

In [ ]:
#Clicking both 'Reviews' & 'Compare Similar' & Warranty' & 'LINKS' (77% of BUYING)
all_clicked = np.array([1,1,1,1]).reshape(1,-1)
print('New visitor - propensity to buy- All Clicked: ', classifier.predict_proba(all_clicked)[:,1].round(2))

# Prediction model is important because it allows the company to offer (or support) the customer based on how likely the customer is to buy - therefore prioritise 'best' customers.